# Masked-Diffusion BabyLM — Strict-Small (Evaluation)

A masked-diffusion denoiser is scored like a masked LM (per-token
pseudo-log-likelihood), so we evaluate it with the official BabyLM **`mlm`**
backend (`strict/` scripts — same as the challenge README).

**Workflow (pick one path):**
- **Quick CSV / sanity check:** Cells 2 → 4 → 5 → 10 (zero-shot on `main` only).
- **Full challenge submission:** Cells 2 → 4 → 5 → 6 → 7 → 9 → 10 (adds fast
  eval on all `chck_*`, GLUE, collate zip).

Cell 4 patches the Hub repo (tokenizer class + latest `model.py`) so eval works.
Always **`git pull` in Cell 2** before evaluating.

The model must already be on the Hub (see notebook **2**).

In [ ]:
# Cell 1 — Parameters
# Set MODEL_ID MANUALLY to the Hub repo you want to evaluate. The next cell lists
# your public models on the Hub (huggingface.co/amosluna) so you can copy one here.
MODEL_ID = "amosluna/babylm-2026-strict-small-mdlm-seed42"
BACKEND  = "mlm"            # masked-diffusion is scored as a masked LM
TRACK    = "strict-small"
EVAL_REPO = "https://github.com/Amos-Luna/Masked-Diffusion-BabyLM.git"
DRIVE_EVAL_ROOT = "/content/drive/MyDrive/Researchs/BabyLM_diffusion_G4/results"  # where results are persisted

In [ ]:
# Cell 2 — Setup: mount Drive, clone/pull repo, then one script does the rest
# (deps, torchvision/timm cleanup, eval data, EWoK fast unzip + gated full set).
# Details live in diffusion/scripts/eval_setup.py.
import os, subprocess, sys
from google.colab import drive; drive.mount("/content/drive")
if not os.path.isdir("/content/Masked-Diffusion-BabyLM"):
    subprocess.run(["git", "clone", EVAL_REPO, "/content/Masked-Diffusion-BabyLM"], check=True)
subprocess.run(["git", "-C", "/content/Masked-Diffusion-BabyLM", "pull"], check=True)
%cd /content/Masked-Diffusion-BabyLM/strict
# Silence TF/HF/protobuf noise in THIS kernel so every later `!` cell inherits it.
sys.path.insert(0, "/content/Masked-Diffusion-BabyLM/diffusion/scripts")
from eval_setup import apply_quiet_env; apply_quiet_env()
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Set HF_TOKEN in Colab Secrets.", e)
!python ../diffusion/scripts/eval_setup.py

In [ ]:
# Cell 3 — List your models + their checkpoint revisions on the Hub (manual pick)
# Lists every model under huggingface.co/amosluna and the chck_NM revisions each
# one exposes, so you can copy the right repo into MODEL_ID (Cell 1) by hand.
from huggingface_hub import HfApi

api = HfApi(token=os.environ.get("HF_TOKEN"))
print("Models on huggingface.co/amosluna:\n")
for m in api.list_models(author="amosluna"):
    try:
        revs = sorted(r.name for r in api.list_repo_refs(m.id).branches if r.name.startswith("chck_"))
    except Exception:
        revs = []
    print(f"  {m.id}")
    if revs:
        print(f"      revisions: {', '.join(revs)}")
print(f"\n>> Currently selected MODEL_ID: {MODEL_ID}")

In [ ]:
# Cell 4 — One-time fix: make the uploaded repo loadable by the eval pipeline.
# Patches main + every chck_* revision (tiny text files only; weights are NOT
# re-uploaded, so it runs in seconds). Three fixes:
#   (1) tokenizer_config.json -> tokenizer_class="PreTrainedTokenizerFast"
#       (models saved by transformers>=5 use "TokenizersBackend", which the eval
#        pipeline's transformers 4.51.x can't resolve -> AutoProcessor error).
#   (2) re-pushes the latest model.py/config.py so trust_remote_code fixes reach
#       old checkpoints (e.g. forward() now tolerates token_type_ids, which the
#       official `reading` task passes -> was crashing with a TypeError).
#   (3) config.json: adds AutoModel -> MaskedDiffusionModel to auto_map. The
#       GLUE finetune harness (Cell 7) loads encoders via AutoModel and needs
#       last_hidden_state; without this it fails with "Unrecognized
#       configuration class ... for this kind of AutoModel".
# Idempotent and safe to re-run. Requires a WRITE HF token (set in Cell 2).
%cd /content/Masked-Diffusion-BabyLM/diffusion
!python scripts/fix_hub_for_eval.py --repo-id $MODEL_ID
%cd /content/Masked-Diffusion-BabyLM/strict

In [ ]:
# Cell 5 — FULL zero-shot on the final checkpoint (main)
# Official eval: BLiMP, supplement, EWoK, entity_tracking, COMPS, reading.
# Enough for a results_summary.csv via Cell 10 (skip Cells 6–7 if time-limited).
!bash scripts/eval_zero_shot.sh $MODEL_ID $BACKEND

In [ ]:
# Cell 6 — (Optional, slow) FAST zero-shot on ALL chck_NM checkpoints
# Required for challenge submission (Cell 9) and the words-seen learning curve.
# Loops chck_1M..chck_10M then chck_20M..chck_100M. Skip if you only need main.
!bash scripts/eval_zero_shot_fast_all_revisions.sh $MODEL_ID $BACKEND $TRACK

In [ ]:
# Cell 7 — GLUE fine-tuning on the final checkpoint (full eval requirement)
# Uses named args; writes per-task scores under strict/results/.
!bash scripts/eval_finetuning.sh --model_path $MODEL_ID --seed 42

In [ ]:
# Cell 8 — (Optional) diffusion-native scorer: ELBO and the layer-duplication
# variant, which the official pipeline cannot express. Writes predictions.json
# in the same layout so it is interchangeable with the mlm backend above.
%cd /content/Masked-Diffusion-BabyLM/diffusion
!python scripts/diffusion_eval_backend.py --model_path_or_name $MODEL_ID \
    --task blimp --data_path ../strict/evaluation_data/full_eval/blimp_filtered \
    --scoring elbo --layer_duplication_factor 2 --backend mlm --save_predictions

In [ ]:
# Cell 9 — Build the submission file (requires Cells 5 + 6 + 7 completed)
# collate_preds.sh passes --fast. If you skipped Cell 6/7, expect "missing" warnings.
%cd /content/Masked-Diffusion-BabyLM/strict
!bash scripts/collate_preds.sh $MODEL_ID $BACKEND $TRACK

In [ ]:
# Cell 10 — Results table + append-only archive to Drive
# (works after Cell 5 alone; finetune/zip included if Cells 7/9 ran).
# Parsing + persistence logic lives in diffusion/scripts/collect_eval_results.py:
#   {DRIVE_EVAL_ROOT}/{model}/{YYYY-MM-DD_HHMMSS}/
#       eval_meta.json | results_summary.csv | results/ | *.zip
%cd /content/Masked-Diffusion-BabyLM/strict
!python ../diffusion/scripts/collect_eval_results.py \
    --model-id $MODEL_ID --backend $BACKEND --track $TRACK \
    --drive-root $DRIVE_EVAL_ROOT